In [1]:
import matplotlib.pyplot as plt
import pathlib, os, random
import numpy as np
import pandas as pd
import tensorflow as tf
import keras


from keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Activation, BatchNormalization, Dropout , GlobalAveragePooling2D
from keras.preprocessing.image import ImageDataGenerator
from keras import Sequential
from keras.callbacks import Callback, EarlyStopping,ModelCheckpoint

In [2]:
#!pip install pandas

In [3]:
train_path="C:\\recent caltech\\journal 1\\indian bird dataset\\mmep\\various split\\7030\\train\\"
no_birds_classes = os.listdir(train_path)
len(no_birds_classes)

41

In [4]:
data_dir = pathlib.Path("C:\\recent caltech\\journal 1\\indian bird dataset\\mmep\\various split\\7030\\train\\")
BirdClasses = np.array(([item.name for item in data_dir.glob("*")]))
print(BirdClasses)

['babblers bird' 'barbets bird' 'bulbuls bird' 'coots bird' 'cranes bird'
 'cuckoos bird' 'Doves bird' 'Drongos bird' 'Ducks bird' 'Eagles bird'
 'Egrets bird' 'Falcons bird' 'Finches bird' 'Flycatchers bird'
 'Herons bird' 'Hornbills bird' 'Jacanas bird' 'kingfishers bird'
 'munias bird' 'nightjars bird' 'orioles bird' 'owls bird'
 'parakeets bird' 'peafowl bird' 'pesants bird' 'pigeons bird'
 'plovers bird' 'prinias bird' 'robins bird' 'sandpipers bird'
 'shrikes bird' 'storks bird' 'sunbirds bird' 'swallows bird'
 'swamphens bird' 'swifts bird' 'terns bird' 'tits bird' 'wagtails bird'
 'warblers bird' 'woodpeckers bird']


In [5]:
train_dir ="C:\\recent caltech\\journal 1\\indian bird dataset\\mmep\\various split\\7030\\train\\"
test_dir = "C:\\recent caltech\\journal 1\\indian bird dataset\\mmep\\various split\\7030\\test\\"
val_dir = "C:\\recent caltech\\journal 1\\indian bird dataset\\mmep\\various split\\7030\\val\\"

In [6]:
train_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory( train_dir , target_size=(224,224) , batch_size=32 , class_mode = "categorical" ,shuffle=True )

val_data = val_gen.flow_from_directory( val_dir , target_size=(224,224) , batch_size=32 , class_mode = "categorical" , shuffle=True )

test_data = test_gen.flow_from_directory( test_dir , target_size=(224,224) , batch_size=32 , class_mode = "categorical" ,shuffle=False )

Found 4303 images belonging to 41 classes.
Found 902 images belonging to 41 classes.
Found 942 images belonging to 41 classes.


In [7]:
#!pip install transformers --no-deps
#!pip install huggingface-hub tokenizers safetensors
#!pip install regex
#!pip install ipywidgets
#!pip install "huggingface-hub>=0.24.0,<1.0"
#!pip install scikit-learn
#!pip install Pillow

In [8]:
import tensorflow as tf
from transformers import AutoImageProcessor, TFAutoModelForImageClassification
#from transformers import  TFAutoModelForImageClassification
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import os

#from transformers import AutoFeatureExtractor

#processor = AutoFeatureExtractor.from_pretrained("google/vit-base-patch16-224")
from transformers import AutoImageProcessor
image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")








# Configuration
MODEL_NAME = "google/vit-base-patch16-224"  # Hugging Face model
IMAGE_SIZE = (224, 224)  # Input size for the model
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4


C:\Users\coder\new_anaconda\anaconda3\envs\final\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
#!pip install torch --no-deps

In [10]:
from transformers import TFViTForImageClassification


model = TFViTForImageClassification.from_pretrained(
    MODEL_NAME,
    from_pt=True  # Indicates the weights are from PyTorch
)

model.summary()

All PyTorch model weights were used when initializing TFViTForImageClassification.

All the weights of TFViTForImageClassification were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFViTForImageClassification for predictions without further training.


Model: "tf_vi_t_for_image_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vit (TFViTMainLayer)        multiple                  85798656  
                                                                 
 classifier (Dense)          multiple                  769000    
                                                                 
Total params: 86,567,656
Trainable params: 86,567,656
Non-trainable params: 0
_________________________________________________________________


In [11]:
"""
model = TFAutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(train_data.class_indices),
    id2label={i: label for label, i in train_data.class_indices.items()},
    label2id={label: i for label, i in train_data.class_indices.items()},
     ignore_mismatched_sizes=True
)
"""

'\nmodel = TFAutoModelForImageClassification.from_pretrained(\n    MODEL_NAME,\n    num_labels=len(train_data.class_indices),\n    id2label={i: label for label, i in train_data.class_indices.items()},\n    label2id={label: i for label, i in train_data.class_indices.items()},\n     ignore_mismatched_sizes=True\n)\n'

In [12]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Lambda, Dense
from tensorflow.keras.models import Model
from transformers import TFViTModel


vit_model = TFViTModel.from_pretrained("google/vit-base-patch16-224",from_pt=True )

# Define input with shape (224, 224, 3)
input_layer = Input(shape=(224, 224, 3), name='input_image')

# Transpose to channels_first
transposed = Lambda(lambda x: tf.transpose(x, perm=[0, 3, 1, 2]))(input_layer)

# Pass through the ViT model
vit_outputs = vit_model(pixel_values=transposed, training=False)

# Extract the [CLS] token
cls_token = vit_outputs.last_hidden_state[:, 0, :]  # Shape: (batch_size, hidden_size)

# Add custom classification layers
x = Dense(1024, activation='relu')(cls_token)
x = Dense(512, activation='relu')(x)
output = Dense(41, activation='softmax')(x)  # Adjust the number of classes as needed

# Define the final model
Model = Model(inputs=input_layer, outputs=output)


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFViTModel: ['classifier.weight', 'classifier.bias']
- This IS expected if you are initializing TFViTModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFViTModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFViTModel were not initialized from the PyTorch model and are newly initialized: ['vit.pooler.dense.weight', 'vit.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
#mobilenet = tf.keras.applications.inception_v3.InceptionV3( include_top=False , weights="imagenet" , input_shape=(224,224,3))

#mobilenet.summary()

In [14]:
"""Model = Sequential([
    model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(256,activation='relu'),
    BatchNormalization(),
    Dense(200,activation='softmax')
])

Model.build(input_shape=(None, 224, 224, 3))  # Include batch size as None
#Model.summary()
Model.summary()
"""

"Model = Sequential([\n    model,\n    GlobalAveragePooling2D(),\n    BatchNormalization(),\n    Dense(256,activation='relu'),\n    BatchNormalization(),\n    Dense(200,activation='softmax')\n])\n\nModel.build(input_shape=(None, 224, 224, 3))  # Include batch size as None\n#Model.summary()\nModel.summary()\n"

In [15]:
#Model.compile( optimizer="adam", loss="categorical_crossentropy" , metrics=["accuracy"] )
Model.compile(optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, decay=1e-6, momentum=0.9, nesterov=True),
               loss = 'categorical_crossentropy',
              metrics = ['accuracy'])

In [16]:
# Create Callback Checkpoint

#checkpoint_path = "BirdsSpecies_Model_Checkpoint"
#checkpoint_callback = ModelCheckpoint(checkpoint_path,monitor="val_accuracy",save_best_only=True)

callbacks = [EarlyStopping(monitor='val_accuracy' , patience=5 , restore_best_weights=True)]

In [17]:
import scipy
#import scipy.ndimage
#import tensorflow as tf

# Force the global namespace to recognize scipy
#globals()['scipy'] = scipy


history = Model.fit(train_data,epochs=25,batch_size=32 ,steps_per_epoch = len(train_data)
,callbacks=callbacks , validation_data=val_data,validation_steps = len(val_data))




Epoch 1/25
135/135 [==============================] - 168s 1s/step - loss: 1.6370 - accuracy: 0.6888 - val_loss: 0.4518 - val_accuracy: 0.9113
Epoch 2/25
135/135 [==============================] - 129s 957ms/step - loss: 0.3053 - accuracy: 0.9328 - val_loss: 0.3009 - val_accuracy: 0.9268
Epoch 3/25
135/135 [==============================] - 128s 941ms/step - loss: 0.1650 - accuracy: 0.9637 - val_loss: 0.2592 - val_accuracy: 0.9335
Epoch 4/25
135/135 [==============================] - 124s 920ms/step - loss: 0.1012 - accuracy: 0.9795 - val_loss: 0.2468 - val_accuracy: 0.9368
Epoch 5/25
135/135 [==============================] - 126s 922ms/step - loss: 0.0674 - accuracy: 0.9886 - val_loss: 0.2382 - val_accuracy: 0.9412
Epoch 6/25
135/135 [==============================] - 128s 950ms/step - loss: 0.0482 - accuracy: 0.9942 - val_loss: 0.2375 - val_accuracy: 0.9401
Epoch 7/25
135/135 [==============================] - 128s 947ms/step - loss: 0.0366 - accuracy: 0.9951 - val_loss: 0.2387 - va

In [18]:
Model.save("transformer1_7030.h5")

In [19]:
test_gen = ImageDataGenerator(rescale=1./255)
test_data = test_gen.flow_from_directory( test_dir , target_size=(224,224) , batch_size=1 , class_mode = "categorical" ,shuffle=False )
filenames = test_data.filenames
nb_samples = len(filenames)

prediction = Model.predict(test_data ,steps = nb_samples)

Found 942 images belonging to 41 classes.
942/942 [==============================] - 81s 80ms/step


In [20]:
from sklearn.metrics import classification_report, confusion_matrix


y_pred = np.argmax(prediction, axis=1)
#y_pred=np.asarray(prediction)
print('Confusion Matrix')
print(confusion_matrix(test_data.classes, y_pred))
print('Classification Report')
target_names =BirdClasses  
print(classification_report(test_data.classes, y_pred, target_names=BirdClasses))


Confusion Matrix
[[20  0  0 ...  0  0  0]
 [ 0 21  0 ...  0  0  0]
 [ 0  0 23 ...  0  0  0]
 ...
 [ 0  0  0 ... 22  0  0]
 [ 0  0  0 ...  0 22  0]
 [ 0  0  0 ...  0  0 22]]
Classification Report
                  precision    recall  f1-score   support

   babblers bird       1.00      0.87      0.93        23
    barbets bird       0.95      0.91      0.93        23
    bulbuls bird       0.92      1.00      0.96        23
      coots bird       0.95      0.87      0.91        23
     cranes bird       0.96      1.00      0.98        23
    cuckoos bird       0.92      1.00      0.96        23
      Doves bird       0.81      0.91      0.86        23
    Drongos bird       0.92      0.96      0.94        23
      Ducks bird       0.95      0.87      0.91        23
     Eagles bird       1.00      0.87      0.93        23
     Egrets bird       0.96      1.00      0.98        23
    Falcons bird       0.86      0.83      0.84        23
    Finches bird       0.92      1.00      0.96   